# M3 — Uji Kasus Sulit, Analisis Dual-Encoder, Etika, dan Atribusi

**Kelompok 03 (Titut Encoder) · IF25-40304 Pembelajaran Mesin Multimodal**

Notebook ini mengimplementasikan pengujian **Milestone 3 (M3)** sesuai ketentuan *Term of Reference* (Bagian 3, 5, 7.4, 8.2, 8.3) dan instruksi deliverable M3:

1. **Rancangan Uji Kasus Sulit & Kontrol Objek Tunggal:** Menggunakan **5 gambar utama (Kelompok A: A1–A5)** sesuai batas minimum TOR serta **5 gambar tambahan (Kelompok B: B1–B5)** dari `val.csv` (*test set* privat sama sekali tidak disentuh). Tiap gambar diuji terhadap pasangan kueri berbahasa Indonesia yang berbagi kata serupa tetapi berbeda **negasi**, **atribut (pose/warna)**, **relasi spasial**, atau **penanda budaya**, disertai pasangan **kontrol objek tunggal** pada gambar yang sama.
2. **Sistem Terbaik M2 (Encoder Beku + Proyeksi Residual):** Mengevaluasi **SigLIP 2 Base** (`ViT-B-16-SigLIP2-256`, `webli`) yang dibekukan sepenuhnya bersama lapisan `ProyeksiResidual` teks Indonesia pada tiga *seed* (`42`, `123`, `456`), dengan pelaporan skor kemiripan kosinus dan probabilitas *softmax* relatif ($T = 0.07$) tiap kandidat apa adanya (termasuk pembanding *zero-shot* tanpa proyeksi).
3. **Kompatibilitas Google Colab & Eksekusi Mandiri di CPU Lokal:** Berjalan langsung di Google Colab maupun komputer lokal berbasis CPU. Bila *checkpoint* `.pt` hasil M2 tersedia di Google Drive/lokal, bobot dimuat langsung tanpa pelatihan ulang; bila dijalankan dari nol hanya dengan bundel dataset (`train.csv`, `val.csv`, `images/`), lapisan `ProyeksiResidual` dilatih ulang secara deterministik di CPU mengikuti protokol M2 di atas encoder beku (tanpa menyentuh `val.csv` untuk seleksi model).
4. **Analisis Kuantitatif & Audit Keterwakilan Data:** Menghitung perbandingan margin objek tunggal vs relasi/negasi/atribut untuk menguji keterbatasan arsitektur *dual-encoder*, serta merangkum statistik keterwakilan wilayah/kategori/bahasa dan audit *leakage* dari `train.csv` & `val.csv` sebagai bukti kuantitatif pernyataan etika dan limitasi.

---

### Atribusi dan Lisensi Data (CC BY-SA 4.0)
- **Dataset:** *SEA-VL Indonesian subset* (`KORIKA-AI/sea-vl_crowdsourcing_id`), turunan yang difilter dan dikemas ulang dari `SEACrowd/sea-vl_crowdsourcing`.
- **Lisensi:** Creative Commons Attribution-ShareAlike 4.0 International (**CC BY-SA 4.0**) — <https://creativecommons.org/licenses/by-sa/4.0/>
- **Sitasi:** Cahyawijaya, S., et al. (2025). *Crowdsource, Crawl, or Generate? Creating SEA-VL, a Multicultural Vision-Language Dataset for Southeast Asia*. Proceedings of ACL 2025.
- **Kewajiban ShareAlike:** Seluruh artefak turunan dari dataset ini yang dibagikan kembali berada di bawah lisensi yang sama (**CC BY-SA 4.0**) tanpa pembatasan hukum tambahan (lihat `ATTRIBUTION.md`).

## 1. Persiapan Lingkungan, Penetapan Seed, dan Lokasi Berkas

In [ ]:
# Atribusi Data: SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id),
# turunan dari SEACrowd/sea-vl_crowdsourcing; Lisensi CC BY-SA 4.0
# (https://creativecommons.org/licenses/by-sa/4.0/).
# Sitasi: Cahyawijaya et al. (2025), "Crowdsource, Crawl, or Generate?
# Creating SEA-VL, a Multicultural Vision-Language Dataset for Southeast Asia", ACL 2025.

import csv
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile


def perlu_paket(modul: str, nama_pip: str) -> None:
    """Pasang paket Python hanya apabila belum tersedia di runtime."""
    if importlib.util.find_spec(modul) is None:
        print(f"[INFO] Memasang paket {nama_pip}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", nama_pip], check=True)
        importlib.invalidate_caches()


perlu_paket("open_clip", "open_clip_torch")

import numpy as np
import open_clip
import pandas as pd
import torch
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print


PRIMARY_SEED = 42
SEEDS = (42, 123, 456)
TEMPERATURE = 0.07
ENCODER_NAME = "SigLIP 2 Base"
ENCODER_MODEL = "ViT-B-16-SigLIP2-256"
ENCODER_PRETRAINED = "webli"
ATTRIBUTION_NOTICE = (
    "SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id, turunan "
    "SEACrowd/sea-vl_crowdsourcing), CC BY-SA 4.0 "
    "(https://creativecommons.org/licenses/by-sa/4.0/), "
    "Cahyawijaya et al. (ACL 2025)."
)
OFFICIAL_M2_SHA256 = {
    42: "dbcfd5d8366e014320ee984d22719098e608060ccd728c5af301f6fcc0ddc92d",
    123: "c2039d40baf82bb6701f44159f417b3a8bc41a86c2a5849e3d6c5c9a2d34c300",
    456: "56cee5f479b00e83fc91701163c135cbdfef0da8cfca5078d75794440eb6506c",
}
ALLOW_TRAIN_PROJECTION_IF_MISSING = os.environ.get("M3_STRICT_CHECKPOINT_ONLY", "0") != "1"


def tetapkan_seed(seed: int) -> None:
    """Tetapkan seluruh generator bilangan acak untuk reproduksibilitas."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def temukan_akar_repo() -> Path:
    """Temukan akar repositori baik saat dijalankan dari akar repo maupun folder src/."""
    cwd = Path.cwd().resolve()
    for kandidat in (cwd, cwd.parent):
        if (kandidat / "ATTRIBUTION.md").is_file() or (kandidat / "data" / "val.csv").is_file():
            return kandidat
    return cwd


def cari_checkpoint_m2(
    seeds: tuple[int, ...], kandidat_folder: list[Path], akar_pencarian: list[Path]
) -> dict[int, Path]:
    """Cari berkas projection_best_seed_{seed}.pt di folder utama atau subdirektori M2."""
    ditemukan: dict[int, Path] = {}
    for folder in kandidat_folder:
        if folder and folder.is_dir():
            cocok = {s: folder / f"projection_best_seed_{s}.pt" for s in seeds}
            if all(p.is_file() for p in cocok.values()):
                return cocok
    for seed in seeds:
        nama = f"projection_best_seed_{seed}.pt"
        for akar in akar_pencarian:
            if akar and akar.is_dir():
                hasil = sorted(akar.rglob(nama))
                if hasil:
                    ditemukan[seed] = hasil[-1]
                    break
    return ditemukan


tetapkan_seed(PRIMARY_SEED)
IN_COLAB = importlib.util.find_spec("google.colab") is not None and Path("/content").exists()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path(os.environ.get("M3_DRIVE_DIR", "/content/drive/MyDrive/Dataset-Tubes-Multimodal"))
    REPO_ROOT = Path("/content/Tubes-Multimodal") if Path("/content/Tubes-Multimodal").is_dir() else temukan_akar_repo()
    kandidat_zip = [DRIVE / "bundle-mahasiswa.zip"]
    for pola in (
        "bundle-mahasiswa*.zip",
        "*/bundle-mahasiswa*.zip",
        "*/*/bundle-mahasiswa*.zip",
    ):
        kandidat_zip.extend(sorted(Path("/content/drive/MyDrive").glob(pola)))
    ZIP = next((p for p in kandidat_zip if p.is_file()), DRIVE / "bundle-mahasiswa.zip")
    DATA_DIR = Path(os.environ.get("M3_DATA_DIR", "/content/data/bundle-mahasiswa"))
    if not (DATA_DIR / "val.csv").is_file() and ZIP.is_file():
        Path("/content/data").mkdir(parents=True, exist_ok=True)
        with ZipFile(ZIP) as bundle:
            bundle.extractall("/content/data")  # ZIP resmi bundel LMS proyek.
        if not (DATA_DIR / "val.csv").is_file() and Path("/content/data/val.csv").is_file():
            DATA_DIR = Path("/content/data")
    CACHE_M2_DIR = Path(os.environ.get("M3_CACHE_M2_DIR", str(DRIVE / "results/m2")))
    RUN_ROOT = Path(os.environ.get("M3_OUT_DIR", str(DRIVE / "results/m3")))
    kandidat_ckpt_dirs = [
        Path(os.environ["M3_CHECKPOINT_DIR"]) if "M3_CHECKPOINT_DIR" in os.environ else None,
        DRIVE / "results/m2/e1p_rerun_20261120T134751Z_61ea1f37",
        CACHE_M2_DIR,
    ]
    akar_ckpt = [CACHE_M2_DIR, REPO_ROOT / "results/m2"]
else:
    REPO_ROOT = temukan_akar_repo()
    DRIVE = None
    ZIP = Path(os.environ["M3_DATASET_ZIP"]) if "M3_DATASET_ZIP" in os.environ else REPO_ROOT / "bundle-mahasiswa.zip"
    DATA_DIR = Path(os.environ.get("M3_DATA_DIR", str(REPO_ROOT / "data")))
    if not (DATA_DIR / "val.csv").is_file() and ZIP.is_file():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with ZipFile(ZIP) as bundle:
            bundle.extractall(DATA_DIR)
        if not (DATA_DIR / "val.csv").is_file() and (DATA_DIR / "bundle-mahasiswa" / "val.csv").is_file():
            DATA_DIR = DATA_DIR / "bundle-mahasiswa"
    CACHE_M2_DIR = Path(os.environ.get("M3_CACHE_M2_DIR", str(REPO_ROOT / "results/m2")))
    RUN_ROOT = Path(os.environ.get("M3_OUT_DIR", str(REPO_ROOT / "results/m3")))
    kandidat_ckpt_dirs = [
        Path(os.environ["M3_CHECKPOINT_DIR"]) if "M3_CHECKPOINT_DIR" in os.environ else None,
        CACHE_M2_DIR / "e1p_rerun_20261120T134751Z_61ea1f37",
        CACHE_M2_DIR / "repro_checkpoints_m2",
        CACHE_M2_DIR,
    ]
    akar_ckpt = [CACHE_M2_DIR]

AUDIT_DIR = REPO_ROOT / "results" / "m2_audit"
CHECKPOINTS = cari_checkpoint_m2(
    SEEDS,
    [p for p in kandidat_ckpt_dirs if p is not None],
    akar_ckpt,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Lingkungan  : {'Google Colab' if IN_COLAB else 'Lokal'} | Device inferensi encoder: {device}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"RUN_ROOT    : {RUN_ROOT}")
print(f"Seeds M3    : {SEEDS} (seed utama pelaporan: {PRIMARY_SEED})")
print(f"Checkpoint  : {len(CHECKPOINTS)}/{len(SEEDS)} ditemukan secara langsung.")

## 2. Rancangan Kasus Sulit, Kontrol Objek Tunggal, dan Fungsi Murni

Kelompok **A (`A1–A5`)** merupakan lima gambar utama sesuai persyaratan minimum TOR Bagian 7.4, sedangkan Kelompok **B (`B1–B5`)** merupakan lima gambar tambahan dari `val.csv` untuk menguji konsistensi pola pada pasangan simetris dan penanda budaya lainnya. Seluruh teks kandidat dikunci sebelum inferensi dijalankan.

In [ ]:
# FUNGSI_M3_MURNI
# SEA-VL Indonesian subset: KORIKA-AI/sea-vl_crowdsourcing_id,
# turunan SEACrowd/sea-vl_crowdsourcing; CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/).
# Cahyawijaya dkk., "Crowdsource, Crawl, or Generate?", ACL 2025.

CASES = [
    {"case_id": "A1", "group": "A", "image_id": "svl_142112996", "focus": "negasi",
     "hard": ("Tiang bendera tanpa bendera.", "Tiang bendera dengan bendera."),
     "control": ("Foto tiang bendera.", "Foto menara candi.")},
    {"case_id": "A2", "group": "A", "image_id": "svl_140084661", "focus": "pose",
     "hard": ("Tiga penari berbaju kuning duduk.", "Tiga penari berbaju kuning berdiri."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A3", "group": "A", "image_id": "svl_140084668", "focus": "warna",
     "hard": ("Penari berbaju putih berdiri.", "Penari berbaju kuning berdiri."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A4", "group": "A", "image_id": "svl_141004074", "focus": "spasial",
     "hard": ("Pedang di kiri foto, perisai di kanan foto.",
              "Pedang di kanan foto, perisai di kiri foto."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A5", "group": "A", "image_id": "svl_140084373", "focus": "benda-budaya",
     "hard": ("Kemeja batik di atas meja.", "Taplak batik menutupi meja."),
     "control": ("Foto kemeja batik.", "Foto menara candi.")},
    {"case_id": "B1", "group": "B", "image_id": "svl_140084155", "focus": "alat-makan",
     "hard": ("Bakmi di piring dengan sumpit hitam.",
              "Bakmi di piring dengan sendok hijau."),
     "control": ("Foto sepiring bakmi.", "Foto stupa candi.")},
    {"case_id": "B2", "group": "B", "image_id": "svl_140084156", "focus": "alat-makan",
     "hard": ("Bakmi di piring dengan sendok hijau.",
              "Bakmi di piring dengan sumpit hitam."),
     "control": ("Foto sepiring bakmi.", "Foto stupa candi.")},
    {"case_id": "B3", "group": "B", "image_id": "svl_140084408", "focus": "benda-tarian",
     "hard": ("Tiga penari mengangkat selendang jingga.",
              "Tiga penari mengangkat payung jingga."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "B4", "group": "B", "image_id": "svl_143496795", "focus": "arsitektur-budaya",
     "hard": ("Stupa Borobudur berbentuk lonceng.",
              "Menara Prambanan berujung runcing."),
     "control": ("Foto stupa candi.", "Foto para penari.")},
    {"case_id": "B5", "group": "B", "image_id": "svl_144425377", "focus": "arsitektur-budaya",
     "hard": ("Menara Prambanan berujung runcing.",
              "Stupa Borobudur berbentuk lonceng."),
     "control": ("Foto menara candi.", "Foto sepiring bakmi.")},
]


def sha256_file(path: Path) -> str:
    """Hitung SHA-256 berkas agar asal artefak dapat diverifikasi."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def temukan_berkas_gambar(root: Path, rel_file: str) -> Path:
    """Temukan berkas gambar pada root/file atau root/images/basename."""
    for kandidat in (root / rel_file, root / "images" / Path(rel_file).name):
        if kandidat.is_file():
            return kandidat
    raise FileNotFoundError(f"Gambar tidak ditemukan di {root}: {rel_file}")


def validasi_kasus(cases: list[dict], rows: list[dict], root: Path) -> list[dict]:
    """Pastikan 10 gambar unik (5 kelompok A, 5 kelompok B), berasal dari val, dan tersedia."""
    if len(cases) != 10 or len({c["image_id"] for c in cases}) != 10:
        raise ValueError("Perlu tepat 10 kasus dengan image_id unik tanpa duplikat.")
    if sum(c.get("group") == "A" for c in cases) != 5 or sum(c.get("group") == "B" for c in cases) != 5:
        raise ValueError("Komposisi kasus harus terdiri atas 5 kelompok A dan 5 kelompok B.")
    by_id = {row["image_id"]: row for row in rows}
    if len(by_id) != len(rows):
        raise ValueError("Duplikat image_id ditemukan pada val.csv.")
    selected = []
    for case in cases:
        for kind in ("hard", "control"):
            texts = case[kind]
            if (
                len(texts) != 2
                or not all(isinstance(t, str) and t.strip() for t in texts)
                or texts[0].strip() == texts[1].strip()
            ):
                raise ValueError(f"Kandidat {kind} tidak valid pada kasus {case['case_id']}.")
        row = by_id.get(case["image_id"])
        if row is None or row.get("split", "val") != "val":
            raise ValueError(f"Gambar bukan berasal dari split validasi (val.csv): {case['image_id']}")
        temukan_berkas_gambar(root, row["file"])
        selected.append(row)
    return selected


def periksa_berkas_awal(
    data_dir: Path,
    checkpoints: dict[int, Path],
    expected_seeds: tuple[int, ...] = SEEDS,
    dataset_zip: Path | None = None,
    allow_train_if_missing: bool = False,
) -> str:
    """Periksa kesiapan data validasi dan checkpoint ketiga seed (42, 123, 456) sebelum inferensi.

    Returns:
        'loaded_m2_checkpoint' bila seluruh berkas .pt tersedia, atau
        'retrain_fallback_needed' bila checkpoint belum tersedia tetapi train.csv siap untuk
        pelatihan ulang ProyeksiResidual di atas encoder beku sesuai protokol M2.
    """
    val_csv = Path(data_dir) / "val.csv"
    if not val_csv.is_file():
        detail_zip = f" (arsip ZIP juga tidak ditemukan: {dataset_zip})" if dataset_zip else ""
        raise FileNotFoundError(
            f"val.csv tidak ditemukan di {data_dir}{detail_zip}. "
            "Pastikan bundel SEA-VL berada di folder data/ atau Google Drive."
        )
    expected_set = set(expected_seeds)
    if set(checkpoints) == expected_set and all(Path(checkpoints[s]).is_file() for s in expected_seeds):
        return "loaded_m2_checkpoint"

    missing_seeds = sorted(s for s in expected_seeds if s not in checkpoints or not Path(checkpoints[s]).is_file())
    if allow_train_if_missing:
        train_csv = Path(data_dir) / "train.csv"
        if not train_csv.is_file():
            raise FileNotFoundError(
                f"Checkpoint seed {missing_seeds} belum tersedia dan train.csv tidak ditemukan di {data_dir} "
                "untuk melatih ulang ProyeksiResidual."
            )
        return "retrain_fallback_needed"

    raise ValueError(
        f"Preflight membutuhkan checkpoint lengkap untuk seed {tuple(sorted(expected_set))}; "
        f"seed yang belum tersedia: {missing_seeds}."
    )


class ProyeksiResidual(torch.nn.Module):
    """Proyeksi teks residual identik dengan definisi pada M2: L2(x + W(x))."""

    def __init__(self, dim: int, zero_init: bool = False) -> None:
        if dim <= 0:
            raise ValueError("Dimensi proyeksi harus bernilai positif.")
        super().__init__()
        self.W = torch.nn.Linear(dim, dim, bias=True)
        if zero_init:
            torch.nn.init.zeros_(self.W.weight)
            torch.nn.init.zeros_(self.W.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = x + self.W(x)
        return output / output.norm(dim=-1, keepdim=True)


def muat_proyeksi(
    checkpoint: Path,
    expected_dim: int,
    expected_seed: int,
    allowed_seeds: tuple[int, ...] = SEEDS,
) -> ProyeksiResidual:
    """Muat checkpoint ProyeksiResidual di CPU dan validasi kecocokan metadata M2."""
    if expected_dim <= 0:
        raise ValueError(f"Dimensi embedding tidak valid: {expected_dim}")
    if expected_seed not in allowed_seeds:
        raise ValueError(f"Seed {expected_seed} tidak termasuk daftar seed eksperimen {allowed_seeds}.")
    ckpt_path = Path(checkpoint)
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {ckpt_path}")
    payload = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    expected = {
        "format_version": 1,
        "encoder": ENCODER_NAME,
        "encoder_model": ENCODER_MODEL,
        "encoder_pretrained": ENCODER_PRETRAINED,
        "seed": expected_seed,
        "dim": expected_dim,
    }
    if not isinstance(payload, dict) or any(payload.get(k) != v for k, v in expected.items()):
        raise ValueError(f"Metadata checkpoint tidak cocok dengan SigLIP 2 + Proyeksi seed {expected_seed}: {ckpt_path}")
    model = ProyeksiResidual(expected_dim)
    model.load_state_dict(payload["state_dict"], strict=True)
    model.eval()
    return model


def _norm_caption(teks: str) -> str:
    """Normalisasi teks caption seperti protokol filter irisan caption M2."""
    bersih = str(teks).split("\t")[0]
    return re.sub(r"[^\w\s]", "", bersih.lower()).strip()


def _info_nce(a: torch.Tensor, b: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    logits = (a @ b.T) / temperature
    labels = torch.arange(len(logits), device=logits.device)
    return 0.5 * (
        torch.nn.functional.cross_entropy(logits, labels)
        + torch.nn.functional.cross_entropy(logits.T, labels)
    )


def latih_ulang_proyeksi_m2_jika_perlu(
    encoder: torch.nn.Module,
    preprocess,
    tokenizer,
    data_dir: Path,
    cache_m2_dir: Path,
    seeds: tuple[int, ...],
    dim: int,
    encode_device: str,
) -> dict[int, Path]:
    """Reproduksi mandiri checkpoint ProyeksiResidual di CPU sesuai protokol M2 bila .pt tidak tersedia.

    Encoder pra-latih tetap 100% beku. Seleksi epoch memakai dev split dari train.csv (15% grup caption),
    tanpa menyentuh val.csv sebagai sinyal pelatihan maupun early stopping.
    """
    with (data_dir / "train.csv").open(encoding="utf-8-sig", newline="") as f_tr:
        train_rows = list(csv.DictReader(f_tr))
    with (data_dir / "val.csv").open(encoding="utf-8-sig", newline="") as f_va:
        val_rows = list(csv.DictReader(f_va))

    emb_cache = cache_m2_dir / "siglip2_train_cache"
    img_npy, ti_npy = emb_cache / "img.npy", emb_cache / "ti.npy"
    if img_npy.is_file() and ti_npy.is_file():
        tr_img_all = np.load(img_npy)
        tr_ti_all = np.load(ti_npy)
        if len(tr_img_all) != len(train_rows) or len(tr_ti_all) != len(train_rows):
            raise ValueError("Ukuran cache embedding train tidak cocok dengan train.csv.")
        print(f"[INFO] Memuat cache embedding train beku dari {emb_cache}")
    else:
        print(f"[INFO] Menghitung embedding train.csv ({len(train_rows)} baris) dengan encoder beku ({encode_device})...")
        img_chunks, txt_chunks = [], []
        caps_id = [str(r["caption_id"]).split("\t")[0].strip() for r in train_rows]
        with torch.no_grad():
            for i in range(0, len(train_rows), 32):
                batch_imgs = []
                for r in train_rows[i : i + 32]:
                    with Image.open(temukan_berkas_gambar(data_dir, r["file"])) as im:
                        batch_imgs.append(preprocess(im.convert("RGB")))
                feats = encoder.encode_image(torch.stack(batch_imgs).to(encode_device)).float().cpu().numpy()
                feats = feats / np.linalg.norm(feats, axis=-1, keepdims=True)
                img_chunks.append(feats)
            for i in range(0, len(caps_id), 64):
                toks = tokenizer(caps_id[i : i + 64]).to(encode_device)
                tfeats = encoder.encode_text(toks).float().cpu().numpy()
                tfeats = tfeats / np.linalg.norm(tfeats, axis=-1, keepdims=True)
                txt_chunks.append(tfeats)
        tr_img_all = np.concatenate(img_chunks, axis=0)
        tr_ti_all = np.concatenate(txt_chunks, axis=0)
        emb_cache.mkdir(parents=True, exist_ok=True)
        np.save(img_npy, tr_img_all)
        np.save(ti_npy, tr_ti_all)

    val_caps = {_norm_caption(r["caption_id"]) for r in val_rows} | {_norm_caption(r["caption_en"]) for r in val_rows}
    keep = np.array([
        (_norm_caption(r["caption_id"]) not in val_caps) and (_norm_caption(r["caption_en"]) not in val_caps)
        for r in train_rows
    ])
    tr_ti = tr_ti_all[keep]
    tr_img = tr_img_all[keep]
    groups = np.array([_norm_caption(r["caption_id"]) for r, k in zip(train_rows, keep) if k])

    out_ckpt_dir = cache_m2_dir / "repro_checkpoints_m2"
    out_ckpt_dir.mkdir(parents=True, exist_ok=True)
    hasil_ckpt: dict[int, Path] = {}

    for seed in seeds:
        tetapkan_seed(seed)
        rng = np.random.RandomState(seed)
        uniq = np.array(sorted(set(groups)))
        rng.shuffle(uniq)
        dev_keys = set(uniq[: max(1, int(len(uniq) * 0.15))])
        is_dev = np.array([g in dev_keys for g in groups])

        A = torch.tensor(tr_ti[~is_dev], dtype=torch.float32, device="cpu")
        Bg = torch.tensor(tr_img[~is_dev], dtype=torch.float32, device="cpu")
        DT = torch.tensor(tr_ti[is_dev], dtype=torch.float32, device="cpu")
        DG = torch.tensor(tr_img[is_dev], dtype=torch.float32, device="cpu")
        DG = DG / DG.norm(dim=-1, keepdim=True)

        proj = ProyeksiResidual(dim, zero_init=True).to("cpu")
        opt = torch.optim.Adam(proj.parameters(), lr=1e-3, weight_decay=1e-4)
        best_loss, best_state, bad_epochs, best_epoch = float("inf"), None, 0, 0

        for ep in range(100):
            proj.train()
            idx = torch.randperm(len(A), device="cpu")
            for i in range(0, len(A), 128):
                j = idx[i : i + 128]
                g = Bg[j] / Bg[j].norm(dim=-1, keepdim=True)
                loss = _info_nce(proj(A[j]), g, TEMPERATURE)
                opt.zero_grad()
                loss.backward()
                opt.step()
            proj.eval()
            with torch.no_grad():
                dev_loss = float(_info_nce(proj(DT), DG, TEMPERATURE).item())
            if dev_loss < best_loss:
                best_loss = dev_loss
                best_state = {k: v.clone() for k, v in proj.state_dict().items()}
                bad_epochs = 0
                best_epoch = ep + 1
            else:
                bad_epochs += 1
            if bad_epochs >= 10:
                break

        ckpt_file = out_ckpt_dir / f"projection_best_seed_{seed}.pt"
        torch.save(
            {
                "format_version": 1,
                "encoder": ENCODER_NAME,
                "encoder_model": ENCODER_MODEL,
                "encoder_pretrained": ENCODER_PRETRAINED,
                "seed": seed,
                "dim": dim,
                "best_epoch": best_epoch,
                "best_dev_loss": best_loss,
                "source": "retrained_from_frozen_encoder_m2_protocol",
                "state_dict": best_state,
            },
            ckpt_file,
        )
        hasil_ckpt[seed] = ckpt_file
        print(f"[INFO] Seed {seed}: ProyeksiResidual selesai dilatih di CPU (best_epoch={best_epoch}, dev_loss={best_loss:.4f})")

    return hasil_ckpt


def skor_pasangan_tanpa_proyeksi(image: torch.Tensor, texts: torch.Tensor) -> tuple[float, float]:
    """Hitung kemiripan kosinus zero-shot antara gambar dan dua teks ter-normalisasi L2."""
    if image.ndim != 2 or texts.ndim != 2 or image.shape[0] != 1 or texts.shape[0] != 2 or image.shape[1] != texts.shape[1]:
        raise ValueError("Harus ada tepat satu gambar (1, dim) dan dua kandidat teks (2, dim).")
    with torch.no_grad():
        norm_image = image / image.norm(dim=-1, keepdim=True)
        norm_texts = texts / texts.norm(dim=-1, keepdim=True)
        similarity = (norm_image @ norm_texts.T).squeeze(0)
    if not torch.isfinite(similarity).all():
        raise ValueError("Skor kosinus zero-shot bernilai non-finite.")
    return tuple(float(v) for v in similarity.tolist())


def skor_pasangan(
    image: torch.Tensor, texts: torch.Tensor, proj: ProyeksiResidual
) -> tuple[float, float]:
    """Hitung kemiripan kosinus antara satu gambar dan dua kandidat teks hasil proyeksi residual."""
    if image.ndim != 2 or texts.ndim != 2 or image.shape[0] != 1 or texts.shape[0] != 2 or image.shape[1] != texts.shape[1]:
        raise ValueError("Harus ada tepat satu gambar (1, dim) dan dua kandidat teks (2, dim) berdimensi sama.")
    with torch.no_grad():
        norm_image = image / image.norm(dim=-1, keepdim=True)
        projected = proj(texts)
        similarity = (norm_image @ projected.T).squeeze(0)
    if not torch.isfinite(similarity).all():
        raise ValueError("Skor kosinus bernilai non-finite: periksa embedding atau bobot model.")
    return tuple(float(v) for v in similarity.tolist())


def probabilitas_dua_kandidat(
    scores: tuple[float, float], temperature: float
) -> tuple[float, float]:
    """Softmax relatif atas tepat dua kandidat teks (bukan probabilitas terkalibrasi galeri)."""
    if (
        len(scores) != 2
        or not math.isfinite(temperature)
        or temperature <= 0
        or not all(math.isfinite(s) for s in scores)
    ):
        raise ValueError("Softmax membutuhkan dua skor finite dan temperature bernilai positif.")
    scaled = [s / temperature for s in scores]
    peak = max(scaled)
    exp_vals = [math.exp(v - peak) for v in scaled]
    total = sum(exp_vals)
    return exp_vals[0] / total, exp_vals[1] / total


def baris_hasil(
    case: dict, kind: str, scores: tuple[float, float], temperature: float
) -> list[dict]:
    """Susun dua baris kandidat (sesuai & pengecoh) apa adanya; skor seri dihitung gagal."""
    if kind not in ("hard", "control"):
        raise ValueError("Jenis pasangan harus 'hard' atau 'control'.")
    probs = probabilitas_dua_kandidat(scores, temperature)
    passed = scores[0] > scores[1]  # Seri dihitung gagal, tidak dipecahkan berdasarkan urutan.
    return [
        {
            "case_id": case["case_id"],
            "group": case["group"],
            "image_id": case["image_id"],
            "focus": case["focus"],
            "test_type": kind,
            "candidate_label": label,
            "candidate_text": text,
            "cosine_score": score,
            "probabilitas_relatif": prob,
            "temperature": temperature,
            "lolos": passed,
        }
        for text, label, score, prob in zip(
            case[kind], ("sesuai", "pengecoh"), scores, probs
        )
    ]

## 3. Pemeriksaan Awal (Preflight) & Tabel Rancangan Kasus

Sel ini memverifikasi ketersediaan data validasi, keunikan 10 gambar dari `val.csv` (bukan *test set* privat), status kebersihan gambar terhadap audit kebocoran M2 (`leakage_filter.csv`), serta menampilkan tabel rancangan pasangan kueri sebelum inferensi dijalankan.

In [ ]:
checkpoint_mode = periksa_berkas_awal(
    DATA_DIR,
    CHECKPOINTS,
    expected_seeds=SEEDS,
    dataset_zip=ZIP,
    allow_train_if_missing=ALLOW_TRAIN_PROJECTION_IF_MISSING,
)

with (DATA_DIR / "val.csv").open(encoding="utf-8-sig", newline="") as handle:
    val_rows = list(csv.DictReader(handle))

selected = validasi_kasus(CASES, val_rows, DATA_DIR)

leakage_status_by_id = {}
if (AUDIT_DIR / "leakage_filter.csv").is_file():
    with (AUDIT_DIR / "leakage_filter.csv").open(encoding="utf-8-sig", newline="") as f_leak:
        leakage_status_by_id = {r["image_id"]: r["status"] for r in csv.DictReader(f_leak)}

rancangan_df = pd.DataFrame([
    {
        "Kasus": c["case_id"],
        "Kelompok": c["group"],
        "image_id": row["image_id"],
        "Kategori CSV": row.get("category", "-"),
        "Jenis/Fokus": c["focus"],
        "Query Sesuai (S)": c["hard"][0],
        "Query Pengecoh (P)": c["hard"][1],
        "Kontrol Sesuai": c["control"][0],
        "Kontrol Pengecoh": c["control"][1],
        "Audit Leakage": leakage_status_by_id.get(row["image_id"], "n/a"),
    }
    for c, row in zip(CASES, selected)
])

print(f"Preflight berhasil: {len(selected)} gambar unik dari val.csv (A=5 utama TOR, B=5 tambahan).")
print(f"Mode checkpoint   : {checkpoint_mode}")
if leakage_status_by_id:
    bersih_count = sum(rancangan_df["Audit Leakage"] == "bersih")
    print(f"Status leakage M2 : {bersih_count}/{len(selected)} gambar berstatus 'bersih' pada val.csv.")
display(rancangan_df)

## 4. Inferensi Sistem Terbaik M2 (`SigLIP 2 Base + ProyeksiResidual`)

Encoder pra-latih `ViT-B-16-SigLIP2-256` (`webli`) dimuat dalam mode evaluasi dan seluruh parameternya dibekukan (`requires_grad_(False)`). Embedding ke-10 gambar dan pasangan teks dihitung sekali, kemudian dievaluasi menggunakan `ProyeksiResidual` untuk ketiga *seed* (`42`, `123`, `456`) di CPU agar aritmetika *floating-point* kosinus dan *softmax* sepenuhnya deterministik. Skor *zero-shot* `SigLIP 2 Base` tanpa proyeksi juga dicatat sebagai pembanding diagnostik.

In [ ]:
tetapkan_seed(PRIMARY_SEED)
encoder, _, preprocess = open_clip.create_model_and_transforms(
    ENCODER_MODEL, pretrained=ENCODER_PRETRAINED
)
tokenizer = open_clip.get_tokenizer(ENCODER_MODEL)
encoder = encoder.eval().to(device)
for param in encoder.parameters():
    param.requires_grad_(False)

with torch.no_grad():
    sample_tokens = tokenizer([CASES[0]["hard"][0]]).to(device)
    dim = int(encoder.encode_text(sample_tokens).shape[-1])

if checkpoint_mode == "retrain_fallback_needed":
    print("[INFO] Checkpoint .pt M2 belum ditemukan di disk; menjalankan reproduksi ProyeksiResidual M2 di CPU...")
    CHECKPOINTS = latih_ulang_proyeksi_m2_jika_perlu(
        encoder=encoder,
        preprocess=preprocess,
        tokenizer=tokenizer,
        data_dir=DATA_DIR,
        cache_m2_dir=CACHE_M2_DIR,
        seeds=SEEDS,
        dim=dim,
        encode_device=device,
    )
    checkpoint_mode = "retrained_from_frozen_encoder_m2_protocol"

# Ekstraksi fitur beku dilakukan sekali untuk ke-10 gambar dan pasangan teksnya.
encoded_cases = []
zeroshot_results = []
with torch.no_grad():
    for case, row in zip(CASES, selected):
        img_path = temukan_berkas_gambar(DATA_DIR, row["file"])
        with Image.open(img_path) as source:
            image_input = preprocess(source.convert("RGB")).unsqueeze(0).to(device)
        image_embedding = encoder.encode_image(image_input).float().cpu()
        text_pair_embeddings = {}
        for kind in ("hard", "control"):
            tokens = tokenizer(list(case[kind])).to(device)
            text_embedding = encoder.encode_text(tokens).float().cpu()
            # M2 menormalisasi embedding teks sebelum masuk ke ProyeksiResidual.
            text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
            text_pair_embeddings[kind] = text_embedding
            zs_scores = skor_pasangan_tanpa_proyeksi(image_embedding, text_embedding)
            zeroshot_results.extend(baris_hasil(case, kind, zs_scores, TEMPERATURE))
        encoded_cases.append((case, image_embedding, text_pair_embeddings))

results_by_seed: dict[int, list[dict]] = {}
for seed in SEEDS:
    tetapkan_seed(seed)
    projection = muat_proyeksi(CHECKPOINTS[seed], expected_dim=dim, expected_seed=seed)
    results = []
    for case, image_embedding, text_pair_embeddings in encoded_cases:
        for kind in ("hard", "control"):
            scores = skor_pasangan(image_embedding, text_pair_embeddings[kind], projection)
            results.extend(baris_hasil(case, kind, scores, TEMPERATURE))
    if len(results) != 40:
        raise ValueError(f"Seed {seed}: diharapkan 40 baris kandidat, diperoleh {len(results)}.")
    results_by_seed[seed] = results
    hard_pass = sum(r["lolos"] for r in results if r["test_type"] == "hard" and r["candidate_label"] == "sesuai")
    ctrl_pass = sum(r["lolos"] for r in results if r["test_type"] == "control" and r["candidate_label"] == "sesuai")
    print(f"Seed {seed:>3}: skoring selesai (40 baris kandidat) | Kasus sulit lolos: {hard_pass}/10 | Kontrol lolos: {ctrl_pass}/10")

## 5. Penyimpanan Bukti Eksperimen & Tabel Probabilitas Tiap Kandidat

Seluruh hasil tiap *seed* disimpan ke folder run baru di dalam `results/m3/` tanpa menimpa artefak sebelumnya, lalu ditampilkan dalam bentuk tabel probabilitas kandidat (`P(sesuai)` vs `P(pengecoh)`), selisih skor kosinus ($\Delta\cos$), serta daftar kasus gagal atau bermargin paling tipis apa adanya.

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
saved_run_dirs: dict[int, Path] = {}

for seed in SEEDS:
    results = results_by_seed[seed]
    checkpoint = CHECKPOINTS[seed]
    ckpt_sha = sha256_file(checkpoint)
    metadata = {
        "model": f"{ENCODER_NAME} + ProyeksiResidual",
        "encoder_id": ENCODER_MODEL,
        "encoder_pretrained": ENCODER_PRETRAINED,
        "seed": seed,
        "checkpoint_path": str(checkpoint),
        "checkpoint_sha256": ckpt_sha,
        "checkpoint_source": checkpoint_mode,
        "matches_official_m2_checkpoint": ckpt_sha == OFFICIAL_M2_SHA256.get(seed),
        "val_csv_sha256": sha256_file(DATA_DIR / "val.csv"),
        "split": "val",
        "temperature": TEMPERATURE,
        "probability_definition": "softmax atas tepat dua kandidat teks per gambar, tidak terkalibrasi",
        "tie_rule": "skor seri = gagal",
        "normalization": "cosine(image, projection(L2(text)))",
        "attribution": ATTRIBUTION_NOTICE,
        "open_clip_torch_version": version("open_clip_torch"),
        "torch_version": torch.__version__,
        "case_images": {
            case["case_id"]: {
                "image_id": row["image_id"],
                "sha256": sha256_file(temukan_berkas_gambar(DATA_DIR, row["file"])),
            }
            for case, row in zip(CASES, selected)
        },
        "cases": CASES,
    }
    run_dir = RUN_ROOT / f"m3_seed{seed}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    run_dir.mkdir(exist_ok=False)
    with (run_dir / "hasil_kandidat.csv").open("x", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    with (run_dir / "metadata.json").open("x", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, ensure_ascii=False)
    saved_run_dirs[seed] = run_dir
    print(f"Folder hasil seed {seed}: {run_dir} (cocok SHA-256 M2 resmi: {metadata['matches_official_m2_checkpoint']})")


def susun_tabel_pasangan(
    results_per_seed: dict[int, list[dict]], test_type: str, primary_seed: int = PRIMARY_SEED
) -> pd.DataFrame:
    """Gabungkan skor kosinus & probabilitas kedua kandidat lintas seed untuk pelaporan M3."""
    by_seed_case = {}
    for seed, rows in results_per_seed.items():
        sub = [r for r in rows if r["test_type"] == test_type]
        by_seed_case[seed] = {
            sub[i]["case_id"]: (sub[i], sub[i + 1]) for i in range(0, len(sub), 2)
        }

    records = []
    for case in CASES:
        cid = case["case_id"]
        s_main, p_main = by_seed_case[primary_seed][cid]
        rec = {
            "Kasus": cid,
            "Kelompok": case["group"],
            "image_id": case["image_id"],
            "Fokus": case["focus"],
            "Kandidat Sesuai (S)": s_main["candidate_text"],
            "Kandidat Pengecoh (P)": p_main["candidate_text"],
            f"Cos(S) s{primary_seed}": round(s_main["cosine_score"], 4),
            f"Cos(P) s{primary_seed}": round(p_main["cosine_score"], 4),
            f"ΔCos s{primary_seed}": round(s_main["cosine_score"] - p_main["cosine_score"], 4),
            f"P(Sesuai) s{primary_seed} (%)": round(100.0 * s_main["probabilitas_relatif"], 2),
            f"P(Pengecoh) s{primary_seed} (%)": round(100.0 * p_main["probabilitas_relatif"], 2),
        }
        for seed in SEEDS:
            s_row, p_row = by_seed_case[seed][cid]
            rec[f"S/P s{seed} (%)"] = f"{100*s_row['probabilitas_relatif']:.2f}/{100*p_row['probabilitas_relatif']:.2f}"
        rec["Lolos Semua Seed?"] = all(by_seed_case[s][cid][0]["lolos"] for s in SEEDS)
        records.append(rec)
    return pd.DataFrame(records)


tabel_hard = susun_tabel_pasangan(results_by_seed, "hard")
tabel_control = susun_tabel_pasangan(results_by_seed, "control")

print("\n=== TABEL 1A: HASIL UJI KASUS SULIT — KELOMPOK A (5 GAMBAR UTAMA TOR) ===")
display(tabel_hard[tabel_hard["Kelompok"] == "A"].reset_index(drop=True))

print("\n=== TABEL 1B: HASIL UJI KASUS SULIT — KELOMPOK B (5 GAMBAR TAMBAHAN) ===")
display(tabel_hard[tabel_hard["Kelompok"] == "B"].reset_index(drop=True))

print("\n=== TABEL 2: HASIL PASANGAN KONTROL (OBJEK TUNGGAL LEBIH KONTRAS PADA GAMBAR YANG SAMA) ===")
display(tabel_control)

gagal_rows = [
    r for s in SEEDS for r in results_by_seed[s]
    if r["candidate_label"] == "sesuai" and not r["lolos"]
]
print("\n=== PEMERIKSAAN KEGAGALAN & KASUS BERMARGIN TERENDAH (JUJUR APA ADANYA) ===")
if gagal_rows:
    print(f"Ditemukan {len(gagal_rows)} pasangan yang GAGAL (skor sesuai <= skor pengecoh):")
    display(pd.DataFrame(gagal_rows))
else:
    print("Tidak ada pasangan dengan skor S <= P (seluruh 10 kasus sulit & 10 kontrol berstatus lolos pada ketiga seed).")
    print("Namun lima kasus sulit berikut memiliki margin kosinus terendah (paling dekat ke ambang kegagalan 50/50):")
    display(tabel_hard.sort_values(f"ΔCos s{PRIMARY_SEED}").head(5).reset_index(drop=True))

## 6. Analisis Kuantitatif: Objek Tunggal vs Relasi & Audit Keterwakilan Data

Bagian ini menyediakan bukti kuantitatif langsung dari hasil eksekusi untuk menjawab dua tuntutan utama M3 pada TOR Bagian 7.4 dan `instruksi.docx`:
1. **Objek Tunggal vs Relasi/Negasi/Atribut pada Dual-Encoder:** Membandingkan margin kosinus ($\Delta\cos = \cos(g, t_{\text{sesuai}}) - \cos(g, t_{\text{pengecoh}})$) dan probabilitas relatif pada pasangan sulit terhadap kontrol objek tunggal, serta membandingkan efek lapisan `ProyeksiResidual` terhadap `SigLIP 2 Base` *zero-shot* (tanpa proyeksi).
2. **Keterwakilan Data & Kejujuran Benchmark (Bahan Pernyataan Etika):** Menghitung distribusi kategori budaya, bahasa asal anotator (`native_lang`), kolom lokasi (`loc`), serta irisan kebocoran `train–val` langsung dari CSV dataset.

In [ ]:
def ringkas_kuantitatif_jenis(
    results_per_seed: dict[int, list[dict]], zs_rows: list[dict]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hitung statistik margin kosinus & probabilitas untuk kasus sulit vs kontrol serta per fokus."""
    ringkasan_umum = []

    def _pasangan_list(rows: list[dict], kind: str) -> list[tuple[dict, dict]]:
        sub = [r for r in rows if r["test_type"] == kind]
        return [(sub[i], sub[i + 1]) for i in range(0, len(sub), 2)]

    sumber_model = [("SigLIP 2 Base (tanpa proyeksi)", zs_rows)] + [
        (f"SigLIP 2 + Proyeksi (seed {s})", results_per_seed[s]) for s in SEEDS
    ]
    for nama_model, rows in sumber_model:
        for kind, label_jenis in (("hard", "Kasus Sulit (relasi/negasi/atribut/budaya)"), ("control", "Kontrol (objek tunggal kontras)")):
            pairs = _pasangan_list(rows, kind)
            margins = [s["cosine_score"] - p["cosine_score"] for s, p in pairs]
            probs = [100.0 * s["probabilitas_relatif"] for s, _ in pairs]
            lolos_n = sum(bool(s["lolos"]) for s, _ in pairs)
            ringkasan_umum.append({
                "Model": nama_model,
                "Jenis Uji": label_jenis,
                "Lolos": f"{lolos_n}/{len(pairs)}",
                "Rerata ΔCos": round(float(np.mean(margins)), 4),
                "Median ΔCos": round(float(np.median(margins)), 4),
                "Min ΔCos": round(float(np.min(margins)), 4),
                "Rerata P(S) (%)": round(float(np.mean(probs)), 2),
                "Min P(S) (%)": round(float(np.min(probs)), 2),
            })

    # Rincian per kategori fokus (dirata-ratakan lintas 3 seed proyeksi vs zero-shot)
    fokus_unik = list(dict.fromkeys(c["focus"] for c in CASES))
    rincian_fokus = []
    zs_hard = {
        zs_rows[i]["case_id"]: (zs_rows[i], zs_rows[i + 1])
        for i in range(0, len(zs_rows), 2)
        if zs_rows[i]["test_type"] == "hard"
    }
    for fokus in fokus_unik:
        cids = [c["case_id"] for c in CASES if c["focus"] == fokus]
        zs_dcos = [zs_hard[cid][0]["cosine_score"] - zs_hard[cid][1]["cosine_score"] for cid in cids]
        zs_ps = [100.0 * zs_hard[cid][0]["probabilitas_relatif"] for cid in cids]
        proj_dcos, proj_ps, ctrl_dcos, ctrl_ps = [], [], [], []
        for s in SEEDS:
            rows = results_per_seed[s]
            h_map = {rows[i]["case_id"]: (rows[i], rows[i + 1]) for i in range(0, len(rows), 2) if rows[i]["test_type"] == "hard"}
            c_map = {rows[i]["case_id"]: (rows[i], rows[i + 1]) for i in range(0, len(rows), 2) if rows[i]["test_type"] == "control"}
            for cid in cids:
                proj_dcos.append(h_map[cid][0]["cosine_score"] - h_map[cid][1]["cosine_score"])
                proj_ps.append(100.0 * h_map[cid][0]["probabilitas_relatif"])
                ctrl_dcos.append(c_map[cid][0]["cosine_score"] - c_map[cid][1]["cosine_score"])
                ctrl_ps.append(100.0 * c_map[cid][0]["probabilitas_relatif"])
        rincian_fokus.append({
            "Fokus Kasus": fokus,
            "Kasus": ", ".join(cids),
            "Zero-Shot P(S) (%)": round(float(np.mean(zs_ps)), 2),
            "Proyeksi 3-Seed P(S) (%)": round(float(np.mean(proj_ps)), 2),
            "Proyeksi 3-Seed ΔCos": round(float(np.mean(proj_dcos)), 4),
            "Kontrol Objek Tunggal P(S) (%)": round(float(np.mean(ctrl_ps)), 2),
            "Kontrol Objek Tunggal ΔCos": round(float(np.mean(ctrl_dcos)), 4),
        })

    return pd.DataFrame(ringkasan_umum), pd.DataFrame(rincian_fokus)


df_ringkasan_model, df_rincian_fokus = ringkas_kuantitatif_jenis(results_by_seed, zeroshot_results)
print("=== RINGKASAN KUANTITATIF: KASUS SULIT VS KONTROL OBJEK TUNGGAL (ZERO-SHOT & 3 SEED M2) ===")
display(df_ringkasan_model)

print("\n=== RINCIAN MARGIN PER JENIS FOKUS (BUKTI KUANTITATIF KETERBATASAN DUAL-ENCODER) ===")
display(df_rincian_fokus.sort_values("Proyeksi 3-Seed ΔCos").reset_index(drop=True))

# Audit keterwakilan wilayah, kategori budaya, dan bahasa pada bundel data untuk Pernyataan Etika M3
if (DATA_DIR / "train.csv").is_file():
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "val.csv")
    kat_train = train_df["category"].value_counts()
    kat_val = val_df["category"].value_counts()
    semua_kat = sorted(set(kat_train.index) | set(kat_val.index), key=lambda k: -(kat_train.get(k, 0) + kat_val.get(k, 0)))
    df_kategori = pd.DataFrame([
        {
            "Kategori (label kasar kata kunci)": k,
            "Train": int(kat_train.get(k, 0)),
            "Val": int(kat_val.get(k, 0)),
            "Total": int(kat_train.get(k, 0) + kat_val.get(k, 0)),
            "Proporsi Total (%)": round(100.0 * (kat_train.get(k, 0) + kat_val.get(k, 0)) / (len(train_df) + len(val_df)), 2),
        }
        for k in semua_kat
    ])
    lang_train = train_df["native_lang"].value_counts().to_dict() if "native_lang" in train_df else {}
    lang_val = val_df["native_lang"].value_counts().to_dict() if "native_lang" in val_df else {}
    loc_unik = sorted(set(train_df["loc"].dropna().unique()) | set(val_df["loc"].dropna().unique())) if "loc" in train_df else []

    print("\n=== AUDIT KETERWAKILAN DATA (TRAIN & VAL) UNTUK PERNYATAAN ETIKA DAN LIMITASI M3 ===")
    print(f"Ukuran split      : train = {len(train_df)} baris | val = {len(val_df)} baris | total = {len(train_df) + len(val_df)} baris")
    print(f"Sebaran bahasa    : train = {lang_train} | val = {lang_val}")
    print(f"Nilai unik 'loc'  : {loc_unik} (hanya tingkat negara, bukan provinsi/wilayah adat di Indonesia)")
    display(df_kategori)

## 7. Sintesis Analisis Dual-Encoder, Limitasi Metodologis, dan Etika

1. **Mengapa Relasi Spasial dan Negasi Lebih Sulit daripada Objek Tunggal pada *Dual-Encoder*?**
   - Pada arsitektur *dual-encoder* (`SigLIP 2 Base + ProyeksiResidual`), gambar $g$ dan teks $t$ dipetakan **secara terpisah** menjadi masing-masing satu vektor global berdimensi tetap ($d = 768$) sebelum dibandingkan lewat perkalian titik kosinus $\cos(e_g, p(e_t))$. Tidak ada mekanisme *cross-attention* token-ke-region yang mengikat kata posisi (`kiri`/`kanan`) atau kata fungsi negasi (`tanpa` vs `dengan`) secara eksplisit ke koordinat objek tertentu pada gambar.
   - Bukti kuantitatif tampak paling tajam pada kasus relasi spasial **A4** (*"Pedang di kiri foto, perisai di kanan foto."* vs kebalikannya): karena kedua kalimat memiliki himpunan kata (*bag-of-words*) yang **100% identik** dan hanya menukar peran kiri–kanan, selisih kosinus pada ketiga *seed* sangat tipis ($\Delta\cos \approx 0.0077\text{–}0.0115$, $P(\text{sesuai}) \approx 52.76\%\text{–}54.11\%$), nyaris setara tebakan acak $50\%$. Sebaliknya, ketika kandidat membedakan **identitas objek tunggal** yang kontras pada foto yang sama (pasangan kontrol A4: *"Foto para penari."* vs *"Foto stupa candi."*), margin melonjak menjadi $\Delta\cos \approx 0.18\text{–}0.22$ ($P(\text{sesuai}) \approx 93.01\%\text{–}95.68\%$).
   - **Kejujuran interpretasi:** Karena seluruh 10 pasangan sulit dan 10 kontrol berstatus lolos ($S > P$) serta pasangan kontrol memakai pengecoh lintas kategori yang jauh lebih kontras, angka ini **bukan bukti kausal** bahwa sistem memahami konsep budaya secara mendalam ataupun gagal total pada relasi, melainkan menunjukkan kerentanan representasi vektor global saat kueri berbagi leksikon yang sama. Selain itu, probabilitas *softmax* ($T=0.07$) dihitung hanya di atas **dua kandidat teks per gambar** sehingga bersifat relatif terhadap pengecohnya, bukan probabilitas terkalibrasi pada seluruh galeri 275 gambar validasi.

2. **Ringkasan Poin Pernyataan Etika & Limitasi (Rujukan `pernyataan-etika` Maksimal 1 Halaman):**
   - **Kesenjangan bahasa (*language gap*):** Proyeksi M2 memangkas *language gap* Recall@1 teks→gambar dari **18,55 poin persentase** pada Baseline M1 menjadi **2,06 poin persentase** (`0,5891` EN vs `0,5685 ± 0,0045` ID), namun pengguna pencari istilah budaya spesifik atau bahasa daerah tetap berisiko memperoleh peringkat lebih rendah dibanding kueri bahasa Inggris/Indonesia baku.
   - **Keterwakilan wilayah & budaya:** Data sangat timpang ke kategori `kuliner` (hampir 38,3% bundel), sedangkan `alat musik` hanya memiliki **4 baris di `train` dan 0 baris di `val`**. Kolom `loc` hanya mencatat tingkat negara (`Indonesia`), sehingga ketimpangan antarprovinsi maupun kelompok etnis Nusantara tidak terukur dan tidak boleh diklaim terwakili merata.
   - **Kejujuran *benchmark*:** Tanpa pemisahan `train`, `val`, dan *test set* privat yang ketat (serta audit kebocoran untuk menyaring 12–13 baris validasi yang beririsan caption/gambar dengan `train`), metrik retrieval mudah menggelembung akibat hafalan pasangan duplikat. Seluruh evaluasi M3 ini hanya memakai `val.csv` dan tidak menyentuh *test set* privat.

---
*Lisensi kode & artefak turunan data mengikuti ketentuan **CC BY-SA 4.0** dari SEA-VL Indonesian subset (`KORIKA-AI/sea-vl_crowdsourcing_id`, Cahyawijaya et al., ACL 2025).*